```mermaid
flowchart LR
    A0["00"] --> A1a["01a"] --> A1b["01b"] --> A2["02"] --> A3["03"] --> A4a["04a"] --> A4b["04b"]
    A4b --> A5a["05a"] --> A5b["05b"] --> A6a["06a"] --> A6b["06b"]
    A6b --> A7["07"] --> A8a["08a"] --> A8b["08b"]
    A8b --> A9["09"] --> A10["10"] --> A11["11"] --> A12["12"] 
    
    classDef normal fill:#f8f9fa,stroke:#adb5bd,stroke-width:1px,color:#111;
    classDef done fill:#e8f7f0,stroke:#198754,stroke-width:1.5px,color:#111;
    classDef current fill:#fff3cd,stroke:#ff8c00,stroke-width:2px,color:#111;
    
    class A0,A1a,A1b,A2,A3,A4a,A4b,A5a,A5b,A6a,A6b,A7,A8a,A8b,A9,A10 done;
    class A11 current;
    class A12 normal;
```

# Notebook 11 — Topic Modeling: thematic structure across time

This notebook introduces **topic modeling** as a way to identify recurring thematic structure in a corpus and to track how those themes vary across historical periods. We use a **CPU-friendly classical workflow** based on document-term matrices and Latent Dirichlet Allocation (LDA), then aggregate topic prevalence by `time_bin` to support diachronic interpretation.

The broader methodological question is:

> What kinds of thematic regularities become visible when we model documents as mixtures of latent topics — and how cautiously should we interpret those topics over time?

This notebook emphasizes:
- interpretability over metric chasing
- topic models as **heuristic summaries**, not hidden truths
- explicit model-selection signals
- comparison of topic prevalence across time bins
- representative documents for interpretation
- responsible reporting of instability and limitations

## Learning goals

By the end of this notebook, students should be able to:

- explain what a topic model does and does **not** do
- build a document-term matrix suitable for LDA
- compare a small range of topic-number settings
- interpret topic-word distributions cautiously
- identify representative documents for topics
- estimate topic prevalence across historical time bins
- visualize topic trends over time
- discuss coherence, stability, and usefulness as different evaluation criteria
- write methodological caveats about topic modeling responsibly

## Method note

A topic model does **not** discover objective themes that exist independently in the corpus. Instead, it finds a statistical decomposition of documents into recurring word distributions. This means that topics are shaped by:

- preprocessing choices
- vocabulary filtering
- document length
- corpus composition
- the chosen number of topics

For that reason, we evaluate topics not only by model metrics (such as perplexity) but also by **human usefulness**: are the top words interpretable? do representative documents make sense? do topic trends over time support meaningful historical questions rather than artifacts?

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import re
from collections import Counter
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

from scipy import sparse

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

import matplotlib.pyplot as plt
import seaborn as sns

import spacy
from spacy.tokens import DocBin

In [ ]:
# ------------------------------------------------------------
# PATHS AND CONFIGURATION
# ------------------------------------------------------------
PROJECT_ROOT = Path('.')

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'

ANALYSIS_DIR = PROJECT_ROOT / 'analysis'
FIGURES_DIR = ANALYSIS_DIR / 'figures'
TABLES_DIR = ANALYSIS_DIR / 'tables'
REPORTS_DIR = ANALYSIS_DIR / 'reports'
MODELS_DIR = ANALYSIS_DIR / 'models'
CACHE_DIR = PROJECT_ROOT / 'cache'

DOC_INDEX = TABLES_DIR / 'nb03-doc_index.csv'
SPLIT_DIR = PROCESSED_DIR / 'nb05-corpus-split'

print('DOC_INDEX:', DOC_INDEX)
print('SPLIT_DIR:', SPLIT_DIR)

## Load the canonical document table

We begin from the stable document index produced earlier in the workflow. This gives us document identifiers, filenames, time bins, and other metadata that let us connect topic outputs back to historical context.

In [ ]:
df = pd.read_csv(DOC_INDEX)
df["publication_year"] = pd.to_numeric(df["publication_year"], errors="coerce").astype("Int64")

print('\nDocuments in doc index:', len(df), '\n')
display(df.head())

## Reload the split spaCy corpus from Notebook 05a

Notebook 05a serialized the corpus as multiple split `.spacy` files. We reload them here so later notebooks do not need to re-run the full linguistic annotation pipeline.

### Critical thinking
Why is it useful to separate *expensive preprocessing* from *later modeling notebooks*?

 =============================================== YOUR THOUGHTS HERE ===============================================



---

In [ ]:
spacy_files = sorted(SPLIT_DIR.glob('*.spacy'))
print(f'Found {len(spacy_files)} split files.')

nlp = spacy.load('en_core_web_sm', disable=['ner'])

docs = []
for fp in spacy_files:
    db = DocBin().from_disk(fp)
    docs.extend(list(db.get_docs(nlp.vocab)))

print('Loaded docs:', len(docs))

## Reconstruct document texts

Topic modeling requires document-level text strings. We reconstruct them from the spaCy documents and align them with the canonical index.

Because the corpus consists of long philosophical books, the resulting topics will often reflect **recurring thematic mixtures within books** rather than short, tightly bounded topics. This is acceptable for our course goals, but it is one reason to interpret topic outputs cautiously.

In [ ]:
chunk_rows = []
for doc in docs:
    # Retrieve metadata from the Doc
    pg_id = doc.user_data.get('pg_id')
    title = doc.user_data.get('title')
    pub_year = doc.user_data.get('publication_year')
    time_bin = doc.user_data.get('time_bin')
    chunk_index = doc.user_data.get('chunk_index', 0)
    
    chunk_rows.append({
        'pg_id': pg_id,
        'title': title,
        'publication_year': pub_year,
        'time_bin': time_bin,
        'chunk_index': chunk_index,
        'text': doc.text,
        'n_tokens': len(doc)
    })

chunk_df = pd.DataFrame(chunk_rows)
chunk_df["publication_year"] = pd.to_numeric(chunk_df["publication_year"], errors="coerce").astype("Int64")
print(f"\nCreated chunk DataFrame with {len(chunk_df)} rows.")
display(chunk_df.head())

## Topic-model tokenization choices

Topic models are highly sensitive to preprocessing. In this notebook we use a simple word-level pipeline that:

- lowercases
- removes punctuation and spaces
- removes stopwords
- removes very short tokens
- keeps only alphabetic tokens

This choice is not neutral. Removing stopwords often helps interpretability, but it can also erase stylistic or relational signals.

### Reflection question
Would a topic model built on lemmatized words differ from one built on raw surface forms? Why?

In [ ]:
STOP_WORDS_FILE = Path('./analysis/stop_words_custom.txt')

def load_stopwords(filepath:Path = STOP_WORDS_FILE) -> set:
    """
    Load custom stop words from a plain text file (one per line).
    Lines starting with '#' are ignored (comments).
    """
    if not filepath.exists():
        raise FileNotFoundError(f"Stopword file not found: {filepath}")
    
    stopwords = set()
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith('#'):
                stopwords.add(line.lower())
    return stopwords

CUSTOM_STOPWORDS = load_stopwords()
print(f"\nLoaded {len(CUSTOM_STOPWORDS)} custom stop words.")

STOPWORDS = list(set(ENGLISH_STOP_WORDS) | CUSTOM_STOPWORDS)
print(f"\nStored {len(STOPWORDS)} stop words in `STOPWORDS`.")


In [ ]:
def topic_tokenize(text: str) -> list[str]:
    """Tokenize text for topic modeling.

    This is intentionally simple and CPU-friendly. The function removes
    punctuation, short tokens, stopwords, and tokens without letters.
    """
    toks = []
    for t in nlp.make_doc(str(text).lower()):
        if t.is_space or t.is_punct:
            continue

        tok = t.text.strip()

        if len(tok) < MIN_TOKEN_LEN:
            continue

        if tok in STOPWORDS:
            continue

        if not re.search(r'[a-z]', tok):
            continue

        toks.append(tok)

    return toks

print(topic_tokenize(chunk_df.iloc[0]['text'])[:30])

## Build the document-term matrix for topic modeling

Classical LDA topic models are typically trained on **count matrices** rather than TF–IDF. TF–IDF is often excellent for retrieval and similarity, but LDA is formulated as a probabilistic mixture model over term counts.

We therefore build a sparse **document-term count matrix** and save it for later reuse.

In [ ]:
# Topic-model preprocessing
MIN_DF = 5
MAX_DF = 0.70
MAX_FEATURES = 25000
MIN_TOKEN_LEN = 3

In [ ]:
# 1. Configure the vectoriser ──────────────────────────────────────
# CountVectorizer converts raw text into a document-term matrix (DTM),
# where each row is a document (chunk) and each column is a term count.
cv = CountVectorizer(
    tokenizer=topic_tokenize,   # use our custom tokeniser (lemmatised, filtered, etc.)
    preprocessor=None,          # skip built-in preprocessing — our tokeniser handles it
    token_pattern=None,         # disable default regex pattern (required when using a custom tokeniser)
    lowercase=False,            # don't lowercase — assume topic_tokenize already handles case
    min_df=MIN_DF,              # ignore terms appearing in fewer than MIN_DF documents
    max_df=MAX_DF,              # ignore terms appearing in more than MAX_DF fraction of documents
    max_features=MAX_FEATURES,  # keep only the top N most frequent terms
)

# 2. Build the document-term matrix ─────────────────────────────────
# Convert the text column to a list of strings and fit+transform in one step.
# tqdm wraps the iterable to show a progress bar during tokenisation.
texts = chunk_df['text'].astype(str).tolist()
X = cv.fit_transform(tqdm(texts, desc="Vectorising chunks"))

# Extract the vocabulary (ordered array of term strings matching DTM columns)
terms = np.array(cv.get_feature_names_out())

print(f"DTM shape: {X.shape}")            # (n_chunks, n_terms)
print(f"Vocabulary size: {len(terms)}")

# 3. Cache outputs to disk ──────────────────────────────────────────
# Save the sparse DTM as a .npz file (efficient for large, sparse matrices)
sparse.save_npz(CACHE_DIR / 'nb11-topic_dtm.npz', X)

# Save the vocabulary as a CSV so we can reload it alongside the DTM later
pd.Series(terms, name='term').to_csv(TABLES_DIR / 'nb11-topic_vocab.csv', index=False)
print("Saved DTM + vocabulary.")

## Model selection signals: topic number, perplexity, and topic diversity

Choosing the number of topics is partly a modeling decision and partly a research judgment. No single metric can tell us the ‘correct’ number of topics.

Here we compare several candidate values using:

- **Perplexity**: a model-fit metric (lower is generally better)
- **Topic diversity**: how distinct the topic top-word lists are

Neither metric is sufficient on its own. A model with better perplexity may still produce less interpretable topics.

### Reflection question
If one model has slightly better perplexity but much less interpretable topics, which should we choose in a humanities-oriented NLP workflow?

In [ ]:
# Model selection
TOPIC_CANDIDATES = [5, 8, 10, 12, 15]
LDA_MAX_ITER = 20
RANDOM_STATE = 42

# Final topic interpretation
N_TOP_WORDS = 15
TOP_REP_DOCS = 5

In [ ]:
def topic_diversity(model: LatentDirichletAllocation, terms: np.ndarray, top_n: int = 15) -> float:
    """Compute a simple topic-diversity score.

    Topic diversity = number of unique top words across all topics / total top-word slots.
    Higher values suggest less overlap among topics, though this is only a rough heuristic.
    """
    top_words = []
    for topic in model.components_:
        top_idx = np.argsort(-topic)[:top_n]
        top_words.extend(terms[top_idx])
    return len(set(top_words)) / max(len(top_words), 1)

selection_rows = []
candidate_models = {}

for k in TOPIC_CANDIDATES:
    lda_k = LatentDirichletAllocation(
        n_components=k,
        learning_method='batch',
        max_iter=LDA_MAX_ITER,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    lda_k.fit(X)

    candidate_models[k] = lda_k

    selection_rows.append({
        'n_topics': k,
        'perplexity': float(lda_k.perplexity(X)),
        'topic_diversity': float(topic_diversity(lda_k, terms, top_n=N_TOP_WORDS)),
    })

selection_df = pd.DataFrame(selection_rows).sort_values('n_topics')
display(selection_df)
selection_df.to_csv(TABLES_DIR / 'nb11-topic_model_selection.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.lineplot(data=selection_df, x='n_topics', y='perplexity', marker='o', ax=axes[0])
axes[0].set_title('Perplexity across topic numbers')

sns.lineplot(data=selection_df, x='n_topics', y='topic_diversity', marker='o', ax=axes[1])
axes[1].set_title('Topic diversity across topic numbers')

for ax in axes:
    ax.set_xlabel('Number of topics')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb11-topic_model_selection.png', dpi=200)
plt.show()

## Choose and fit the final LDA model

For teaching purposes, we select the topic number with the lowest perplexity by default, but this should always be checked against interpretability. In a research setting, you would typically inspect several candidate models before committing to one.

In [ ]:
BEST_K = int(selection_df.sort_values('perplexity').iloc[0]['n_topics'])
print('Selected topic count (by lowest perplexity):', BEST_K)

lda = candidate_models[BEST_K]
doc_topic = lda.transform(X)

print('Document-topic matrix shape:', doc_topic.shape)

## Inspect topics: top words and representative documents

Topic interpretation should use at least two sources of evidence:

1. **top words** for each topic
2. **representative documents** with high topic proportions

Top words alone can be misleading. Representative documents help us see whether a topic corresponds to a meaningful theme, a stylistic pattern, or a corpus artifact.

In [ ]:
# --- Get top terms per topic (long format) ---
def get_top_topic_terms(model: LatentDirichletAllocation, terms: np.ndarray, top_n: int = 15) -> pd.DataFrame:
    """Return a tidy dataframe of top words per topic."""
    rows = []
    for topic_i, topic in enumerate(model.components_):
        top_idx = np.argsort(-topic)[:top_n]
        for rank, j in enumerate(top_idx, start=1):
            rows.append({
                'topic': topic_i,
                'rank': rank,
                'term': terms[j],
                'weight': float(topic[j]),
            })
    return pd.DataFrame(rows)

topic_terms = get_top_topic_terms(lda, terms, top_n=N_TOP_WORDS)
display(topic_terms.head(30))
topic_terms.to_csv(TABLES_DIR / 'nb11-topic_top_terms.csv', index=False)

# --- Create a summary table: topic ID + list of top terms ---
# Group by topic, sort by rank, collect terms
topic_summary = (
    topic_terms.sort_values(['topic', 'rank'])
    .groupby('topic')['term']
    .apply(lambda x: ', '.join(x))   # comma-separated string
    .reset_index(name='top_terms')
)

# Optionally, also include the top weights as a separate column
topic_summary_weights = (
    topic_terms.sort_values(['topic', 'rank'])
    .groupby('topic')['weight']
    .apply(lambda x: ', '.join([f'{w:.3f}' for w in x]))
    .reset_index(name='top_weights')
)

# Merge both
topic_summary = topic_summary.merge(topic_summary_weights, on='topic')

print("\nTopic summary (top terms and weights):")
display(topic_summary)

# Save summary
topic_summary.to_csv(TABLES_DIR / 'nb11-topic_summary.csv', index=False)

In [ ]:
doc_topic_df = pd.DataFrame(doc_topic, columns=[f'topic_{i}' for i in range(BEST_K)])
doc_topic_df['dominant_topic'] = doc_topic_df.idxmax(axis=1).str.replace('topic_', '', regex=False).astype(int)

doc_meta = df[['pg_id', 'title', 'time_bin']].copy()
if 'title' in df.columns:
    doc_meta['title'] = df['title']

# doc_topics_full = pd.concat([doc_meta.reset_index(drop=True), doc_topic_df.reset_index(drop=True)], axis=1)
doc_topics_full = doc_meta.merge(doc_topic_df, left_index=True, right_index=True)  # but need to align by index

rep_rows = []
for topic_i in range(BEST_K):
    col = f'topic_{topic_i}'
    top_docs = doc_topics_full.nlargest(TOP_REP_DOCS, col)
    for _, r in top_docs.iterrows():
        rep_rows.append({
            'topic': topic_i,
            'pg_id': r['pg_id'],
            'title': r['title'],
            'time_bin': r['time_bin'],
            'topic_weight': r[col],
        })

rep_docs = pd.DataFrame(rep_rows)
display(rep_docs.head(20))
rep_docs.to_csv(TABLES_DIR / 'nb11-topic_representative_docs.csv', index=False)

## Topic prevalence over time

Each document receives a topic mixture. By averaging those topic proportions within each `time_bin`, we can estimate which topics become more or less prevalent across periods.

This is not the same as proving that a theme historically ‘caused’ a change. It only shows that the statistical topic mixture shifts across the corpus over time.

### Reflection question
If a topic rises sharply in one bin, what alternative explanations should we consider besides ‘the idea became more important’?

In [ ]:
topic_cols = [f'topic_{i}' for i in range(BEST_K)]

topic_time = (
    doc_topics_full.dropna(subset=['time_bin'])
                  .groupby('time_bin')[topic_cols]
                  .mean()
                  .reset_index()
)

display(topic_time.head())
topic_time.to_csv(TABLES_DIR / 'nb11-topic_prevalence_by_time_bin.csv', index=False)

In [ ]:
bin_counts = doc_topics_full.dropna(subset=['time_bin']).groupby('time_bin').size().reset_index(name='n_docs')
display(bin_counts)
bin_counts.to_csv(TABLES_DIR / 'nb11-documents_per_time_bin.csv', index=False)

In [ ]:
# Helper to extract start year from time_bin string
def get_start_year(b: str) -> int:
    try:
        return int(b.split('–')[0] if '–' in b else b.split('-')[0])
    except (ValueError, IndexError, AttributeError):
        return 0

In [ ]:
# --- Reorder time_bin as categorical ---
topic_time_long = topic_time.melt(id_vars='time_bin', var_name='topic', value_name='mean_topic_weight')
topic_time_long['topic'] = topic_time_long['topic'].str.replace('topic_', 'Topic ', regex=False)

# Get unique time bins and sort chronologically
chronological_bins = sorted(topic_time_long['time_bin'].unique(), key=get_start_year)

# Convert to ordered categorical
topic_time_long['time_bin'] = pd.Categorical(
    topic_time_long['time_bin'],
    categories=chronological_bins,
    ordered=True
)

# --- Plot ---
plt.figure(figsize=(14, 7))
sns.lineplot(
    data=topic_time_long,
    x='time_bin',
    y='mean_topic_weight',
    hue='topic',
    marker='o'
)
plt.title('Topic prevalence across time bins')
plt.xlabel('Time bin')
plt.ylabel('Mean document topic weight')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb11-topic_prevalence_over_time.png', dpi=200)
plt.show()

In [ ]:
with pd.option_context('display.max_colwidth', None):
    display(topic_summary[['top_terms']])

In [ ]:
# Prepare heatmap data
topic_heat = topic_time.set_index('time_bin')[topic_cols].T
topic_heat.index = [f'Topic {i}' for i in range(BEST_K)]

# --- Sort columns chronologically ---
chronological_bins = sorted(topic_heat.columns, key=get_start_year)
topic_heat = topic_heat[chronological_bins]   # reorder columns

# --- Plot ---
plt.figure(figsize=(12, 6))
sns.heatmap(topic_heat, cmap='mako')
plt.title('Topic prevalence heatmap by time bin')
plt.xlabel('Time bin')
plt.ylabel('Topic')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb11-topic_prevalence_heatmap.png', dpi=200)
plt.show()

## Inspect a topic in context

A strong habit in topic-model interpretation is to move repeatedly between:

- the **topic word list**
- representative documents
- the historical bins where the topic is more prevalent

This triangulation helps us avoid over-interpreting topics from word lists alone.

In [ ]:
TOPIC_TO_INSPECT = 0

print('Top words for Topic', TOPIC_TO_INSPECT)
display(topic_terms[topic_terms['topic'] == TOPIC_TO_INSPECT])

print('Representative documents for Topic', TOPIC_TO_INSPECT)
display(rep_docs[rep_docs['topic'] == TOPIC_TO_INSPECT])

## Reporting standards and limitations

When reporting topic-model results, you should explicitly state:

- the preprocessing choices used
- the number of topics explored and the final choice
- the signals used for selection (e.g., perplexity, diversity, interpretability)
- whether topics were interpreted with representative documents
- that topic prevalence over time may reflect corpus composition, not only thematic change

### Some reflections
- Which topics were easiest to interpret? Which looked mixed or noisy?
- Did the ‘best’ model by perplexity also produce the most useful topics?
- Which topic trends over time looked historically plausible? Which might reflect artifacts?
- How would chunk-level topic modeling differ from full-document topic modeling in this corpus?

---

## Replication check: topic stability across random seeds

LDA is a stochastic model: even when the corpus and preprocessing remain the same, different random initializations can lead to somewhat different topic solutions. This does not make topic modeling invalid, but it does mean that topics should be treated as **model-dependent summaries** rather than fixed latent truths.

To make this explicit, we run a small replication check. We refit the chosen LDA configuration with **three different random seeds**, extract the **top words** for each topic, and compare topics across runs using **Jaccard similarity** on their top-word sets.

The purpose is not to demand perfect identity across runs. Instead, the goal is methodological: if a topic remains broadly recognizable across replications, we can treat it as more stable. If it changes substantially from one run to another, we should interpret it more cautiously.

In [ ]:
# ------------------------------------------------------------
# Replication check: refit LDA with 3 seeds and compare
# topic agreement using Jaccard overlap on top words
# ------------------------------------------------------------

REPLICATION_SEEDS = [11, 42, 99]
TOP_WORDS_STABILITY = globals().get("N_TOP_WORDS", 15)

# 1) Find the final document-term matrix used for LDA
X_CANDIDATES = ["X_dtm", "X", "dtm", "doc_term_matrix"]
X_topic = None
for name in X_CANDIDATES:
    if name in globals():
        X_topic = globals()[name]
        print(f"Using document-term matrix: {name}")
        break
if X_topic is None:
    raise NameError(
        "Could not find the document-term matrix. Expected one of: "
        f"{X_CANDIDATES}"
    )

# 2) Find the vectorizer so we can recover feature names
VECT_CANDIDATES = ["vectorizer", "count_vectorizer", "cv", "bow_vectorizer"]
vectorizer_topic = None
for name in VECT_CANDIDATES:
    if name in globals():
        vectorizer_topic = globals()[name]
        print(f"Using vectorizer: {name}")
        break
if vectorizer_topic is None:
    raise NameError(
        "Could not find the fitted CountVectorizer. Expected one of: "
        f"{VECT_CANDIDATES}"
    )

feature_names = np.array(vectorizer_topic.get_feature_names_out())

# 3) Find the final topic number
if "FINAL_N_TOPICS" in globals():
    FINAL_N_TOPICS = globals()["FINAL_N_TOPICS"]
elif "N_TOPICS" in globals():
    FINAL_N_TOPICS = globals()["N_TOPICS"]
elif "best_k" in globals():
    FINAL_N_TOPICS = globals()["best_k"]
elif "final_lda" in globals():
    FINAL_N_TOPICS = globals()["final_lda"].n_components
elif "lda" in globals() and hasattr(globals()["lda"], "n_components"):
    FINAL_N_TOPICS = globals()["lda"].n_components
else:
    raise NameError(
        "Could not infer the final number of topics. Define FINAL_N_TOPICS "
        "or ensure your fitted LDA model is available."
    )

# 4) Reuse the notebook's LDA iteration setting if present
LDA_MAX_ITER_LOCAL = globals().get("LDA_MAX_ITER", 20)

print("Final topic number:", FINAL_N_TOPICS)
print("Top words per topic for stability check:", TOP_WORDS_STABILITY)
print("Replication seeds:", REPLICATION_SEEDS)


def get_topic_word_lists(lda_model, feature_names, top_n=15):
    out = []
    for topic_weights in lda_model.components_:
        top_idx = np.argsort(-topic_weights)[:top_n]
        out.append(feature_names[top_idx].tolist())
    return out


def jaccard(words_a, words_b):
    a, b = set(words_a), set(words_b)
    return len(a & b) / len(a | b) if (a or b) else np.nan


# ------------------------------------------------------------------
# Fit replicated LDA runs
# ------------------------------------------------------------------
lda_runs = {}
topic_word_lists = {}

for seed in REPLICATION_SEEDS:
    lda_rep = LatentDirichletAllocation(
        n_components=FINAL_N_TOPICS,
        max_iter=LDA_MAX_ITER_LOCAL,
        learning_method="batch",
        random_state=seed,
    )
    lda_rep.fit(X_topic)
    lda_runs[seed] = lda_rep
    topic_word_lists[seed] = get_topic_word_lists(
        lda_rep,
        feature_names,
        top_n=TOP_WORDS_STABILITY,
    )

print("Replication runs completed.")

# ------------------------------------------------------------------
# Compare each baseline topic to its best-matching topic in each
# additional run using Jaccard similarity on top-word sets
# ------------------------------------------------------------------
baseline_seed = REPLICATION_SEEDS[0]
comparison_rows = []
heatmaps = {}

for other_seed in REPLICATION_SEEDS[1:]:
    sim_matrix = np.zeros((FINAL_N_TOPICS, FINAL_N_TOPICS))

    for i in range(FINAL_N_TOPICS):
        for j in range(FINAL_N_TOPICS):
            sim_matrix[i, j] = jaccard(
                topic_word_lists[baseline_seed][i],
                topic_word_lists[other_seed][j],
            )

    heatmaps[other_seed] = sim_matrix

    for i in range(FINAL_N_TOPICS):
        best_j = int(sim_matrix[i].argmax())
        comparison_rows.append({
            "baseline_seed": baseline_seed,
            "other_seed": other_seed,
            "baseline_topic": i,
            "best_match_topic": best_j,
            "best_jaccard": float(sim_matrix[i, best_j]),
            "baseline_words": ", ".join(topic_word_lists[baseline_seed][i]),
            "matched_words": ", ".join(topic_word_lists[other_seed][best_j]),
        })

replication_df = pd.DataFrame(comparison_rows)
display(
    replication_df[[
        "other_seed", "baseline_topic", "best_match_topic", "best_jaccard"
    ]].sort_values(["other_seed", "baseline_topic"])
)

# ------------------------------------------------------------------
# Summary table by baseline topic
# ------------------------------------------------------------------
replication_summary = (
    replication_df.groupby("baseline_topic", as_index=False)["best_jaccard"]
    .agg(mean_best_jaccard="mean", min_best_jaccard="min", max_best_jaccard="max")
    .sort_values("baseline_topic")
)

overall_mean_jaccard = replication_df["best_jaccard"].mean()
print(f"Overall mean best-match Jaccard: {overall_mean_jaccard:.3f}")
display(replication_summary)

# ------------------------------------------------------------------
# Heatmap for the first comparison run
# ------------------------------------------------------------------
seed_b = REPLICATION_SEEDS[1]
sim_df = pd.DataFrame(
    heatmaps[seed_b],
    index=[f"seed{baseline_seed}_topic{i}" for i in range(FINAL_N_TOPICS)],
    columns=[f"seed{seed_b}_topic{j}" for j in range(FINAL_N_TOPICS)],
)

plt.figure(figsize=(10, 8))
sns.heatmap(sim_df, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0, vmax=1)
plt.title(
    f"Topic agreement by Jaccard overlap on top words\n"
    f"seed {baseline_seed} vs seed {seed_b}"
)
plt.xlabel("Topics in second run")
plt.ylabel("Topics in baseline run")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# Short interpretation aid
# ------------------------------------------------------------------
if overall_mean_jaccard >= 0.60:
    interpretation = (
        "The topic solution looks reasonably stable across these replications: "
        "most baseline topics have fairly similar top-word counterparts in the other runs."
    )
elif overall_mean_jaccard >= 0.40:
    interpretation = (
        "The topic solution shows moderate stability: some topics replicate fairly well, "
        "while others appear more sensitive to random initialization."
    )
else:
    interpretation = (
        "The topic solution appears quite sensitive to initialization: topic interpretation "
        "should be treated cautiously, especially for weaker or less coherent topics."
    )

print("Interpretation note:", interpretation)

## Reflection note

A high best-match Jaccard score means that a topic's top words are fairly similar across random seeds, which suggests a more stable thematic pattern. Lower scores indicate that the topic solution is more sensitive to initialization, so topic interpretation should be more cautious.

This does **not** mean that unstable topics are useless. Rather, it means they should be treated as more tentative summaries of the corpus. In practice, topic modeling is often most trustworthy when both the top words and the broader thematic story remain broadly recognizable across multiple runs.

## Methodological takeaway

This replication check extends the notebook's interpretation-first approach. Topic models can be informative, but they are not uniquely determined by the data. Repeating the fit with different random seeds reminds us that topic outputs are **model-based approximations**, not direct discoveries of objective thematic structure.

For this course, the practical lesson is simple: topic interpretation should be supported not only by top words and representative texts, but also by at least one lightweight stability check. If a topic disappears or changes drastically across replications, it should be reported more cautiously than a topic that remains broadly consistent.

In [ ]:
# ------------------------------------------------------------
# Save reusable outputs for later notebooks / paper writing
# ------------------------------------------------------------
doc_topics_full.to_csv(TABLES_DIR / 'nb11-document_topic_matrix.csv', index=False)

report = {
    'n_documents': int(len(df)),
    'vocabulary_size': int(len(terms)),
    'topic_candidates': TOPIC_CANDIDATES,
    'selected_n_topics': int(BEST_K),
    'lda_max_iter': int(LDA_MAX_ITER),
    'min_df': int(MIN_DF),
    'max_df': float(MAX_DF),
    'max_features': int(MAX_FEATURES),
}

with (REPORTS_DIR / 'nb11-topic_modeling_run_summary.json').open('w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)

print('Saved document-topic matrix and run summary.')

---

```mermaid
flowchart TB
    A0["00<br/>Bootcamp"] --> P1

    subgraph P1["Part I — Corpus building and analysis"]
        direction LR
        A1a["01a<br/>Corpus metadata"] --> A1b["01b<br/>Corpus building"] --> A2["02<br/>Preprocessing"] --> A3["03<br/>Distributions + time"] --> A4a["04a<br/>Lexical exploration"] --> A4b["04b<br/>Embedding"]
    end

    subgraph P2["Part II — Linguistic annotations"]
        direction LR
        A5a["05a<br/>spaCy annotation"] --> A5b["05b<br/>Relation extraction"] --> A6a["06a<br/>NER"] --> A6b["06b<br/>Custom NER"]
    end

    subgraph P3["Part III — Representations"]
        direction LR
        A7["07<br/>BoW + TF-IDF"] --> A8a["08a<br/>Embeddings"] --> A8b["08b<br/>Transformers"]
    end

    subgraph P4["Part IV — Models and interpretation"]
        direction LR
        A9["09<br/>Classification"] --> A10["10<br/>Custom NER training"] --> A11["11<br/>Topic modeling"] --> A12["12<br/>Semantic shift"]
    end

    P1 --> P2
    P2 --> P3
    P3 --> P4

    classDef start fill:#f3f0ff,stroke:#6f42c1,stroke-width:1.5px,color:#111;
    classDef prep fill:#eef7ff,stroke:#1f77b4,stroke-width:1.5px,color:#111;
    classDef annot fill:#eefaf0,stroke:#2ca02c,stroke-width:1.5px,color:#111;
    classDef repr fill:#fff7e6,stroke:#ff8c00,stroke-width:1.5px,color:#111;
    classDef model fill:#fff0f0,stroke:#d62728,stroke-width:1.5px,color:#111;

    classDef highlight fill:#fff3b0,stroke:#f5a623,stroke-width:4px,color:#111;

    class A1a,A1b,A2,A3,A4a,A4b prep;
    class A5a,A5b,A6a,A6b annot;
    class A7,A8a,A8b repr;
    class A9,A10,A11,A12 model;

    class A11 highlight;
```